# 02 · Data cleaning and SQLite storage

Applies the fixes found in `01_data_overview`: **handle missing values, fix types, encode categories**, then save the cleaned data to `data/processed/` and load it into a SQLite database (`heart.db`).

The logic lives in `src/cleaning.py` and `src/database.py` so it can be rerun outside the notebook (`python -m src.cleaning && python -m src.database`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_raw
from src.cleaning import clean, encode, CLEAN_PATH, ENCODED_PATH, PROCESSED_DIR
from src.database import build_database, connect, DB_PATH

pd.set_option("display.max_columns", None)

## 1. Missing values

Only `ca` (4 rows) and `thal` (2 rows) are missing. These are diagnostic test results, so filling them in would invent a clinical finding. Dropping 6 of 303 rows (≈2%) is the simpler, more honest choice. Check first that dropping them doesn't shift the class balance:

In [2]:
raw = load_raw()
missing_rows = raw[raw[["ca", "thal"]].isna().any(axis=1)]
missing_rows

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
87,53.0,0.0,3.0,128.0,216.0,0.0,2.0,115.0,0.0,0.0,1.0,0.0,NaN,0
166,52.0,1.0,3.0,138.0,223.0,0.0,0.0,169.0,0.0,0.0,1.0,NaN,3.0,0
192,43.0,1.0,4.0,132.0,247.0,1.0,2.0,143.0,1.0,0.1,2.0,NaN,7.0,1
266,52.0,1.0,4.0,128.0,204.0,1.0,0.0,156.0,1.0,1.0,2.0,0.0,NaN,2
287,58.0,1.0,2.0,125.0,220.0,0.0,0.0,144.0,0.0,0.4,2.0,NaN,7.0,0
302,38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,NaN,3.0,0


In [3]:
cleaned = clean(raw)
balance = pd.DataFrame({
    "raw": (raw["num"] > 0).value_counts(normalize=True).sort_index(),
    "cleaned": cleaned["target"].value_counts(normalize=True).sort_index(),
}).rename(index={False: "no disease", True: "disease", 0: "no disease", 1: "disease"})
print(f"Rows: {len(raw)} -> {len(cleaned)}")
(balance * 100).round(1)

Rows: 303 -> 297


,raw,cleaned
no disease,54.1,53.9
disease,45.9,46.1


## 2. Types

The `?` markers forced every column to `float64`. With them gone, everything except `oldpeak` is cast back to integer. `clean()` also adds `patient_id` (the line number in the raw file) and `target`, and checks every code against the data dictionary.

In [4]:
cleaned.dtypes

patient_id      int64
age             int64
sex             int64
cp              int64
trestbps        int64
chol            int64
fbs             int64
restecg         int64
thalach         int64
exang           int64
oldpeak       float64
slope           int64
ca              int64
thal            int64
num             int64
target          int64
dtype: object

In [5]:
cleaned.head()

,patient_id,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
0,1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0,0
1,2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2,1
2,3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1,1
3,4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0,0
4,5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0,0


## 3. Encode categories

`sex`, `fbs` and `exang` are already 0/1, and `ca` is a count. The multi-class codes `cp`, `restecg`, `slope` and `thal` are one-hot encoded with readable names so a model's coefficients can be read directly (e.g. `cp_asymptomatic`). `num` is dropped from the model file because `target` is derived from it.

In [6]:
encoded = encode(cleaned)
print(encoded.shape)
encoded.head()

(297, 24)


,patient_id,age,sex,cp_typical_angina,cp_atypical_angina,cp_non_anginal_pain,cp_asymptomatic,trestbps,chol,fbs,restecg_normal,restecg_st_t_abnormality,restecg_lv_hypertrophy,thalach,exang,oldpeak,slope_upsloping,slope_flat,slope_downsloping,ca,thal_normal,thal_fixed_defect,thal_reversible_defect,target
0,1,63,1,1,0,0,0,145,233,1,0,0,1,150,0,2.3,0,0,1,0,0,1,0,0
1,2,67,1,0,0,0,1,160,286,0,0,0,1,108,1,1.5,0,1,0,3,1,0,0,1
2,3,67,1,0,0,0,1,120,229,0,0,0,1,129,1,2.6,0,1,0,2,0,0,1,1
3,4,37,1,0,0,1,0,130,250,0,1,0,0,187,0,3.5,0,0,1,0,1,0,0,0
4,5,41,0,0,1,0,0,130,204,0,0,0,1,172,0,1.4,1,0,0,0,1,0,0,0


## 4. Save to `data/processed/`

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(CLEAN_PATH, index=False)
encoded.to_csv(ENCODED_PATH, index=False)
[p.name for p in sorted(PROCESSED_DIR.iterdir()) if p.suffix == ".csv"]

['heart_clean.csv', 'heart_encoded.csv']

## 5. Load into SQLite

Schema: [`sql/schema.sql`](../sql/schema.sql). A `patients` table (primary key `patient_id`, CHECK constraints on every range) with foreign keys into four lookup tables for the category labels. `target` is a generated column, so it can never disagree with `num`. The `patients_labeled` view joins the labels back in.

In [8]:
build_database(cleaned)
conn = connect()
pd.read_sql("SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name", conn)

,type,name
0,index,idx_patients_target
1,table,chest_pain_types
2,table,patients
3,table,resting_ecg_results
4,table,st_slopes
5,table,thal_results
6,view,patients_labeled


In [9]:
pd.read_sql("SELECT * FROM patients_labeled LIMIT 5", conn)

,patient_id,age,sex,chest_pain_type,trestbps,chol,fasting_blood_sugar_gt_120,resting_ecg,thalach,exercise_angina,oldpeak,st_slope,ca,thal,num,target
0,1,63,male,typical_angina,145,233,true,lv_hypertrophy,150,no,2.3,downsloping,0,fixed_defect,0,0
1,2,67,male,asymptomatic,160,286,false,lv_hypertrophy,108,yes,1.5,flat,3,normal,2,1
2,3,67,male,asymptomatic,120,229,false,lv_hypertrophy,129,yes,2.6,flat,2,reversible_defect,1,1
3,4,37,male,non_anginal_pain,130,250,false,normal,187,no,3.5,downsloping,0,normal,0,0
4,5,41,female,atypical_angina,130,204,false,lv_hypertrophy,172,no,1.4,upsloping,0,normal,0,0


### Constraints at work

The database refuses a chest-pain code that isn't in the lookup table:

In [10]:
bad_row = """
INSERT INTO patients (patient_id, age, sex, cp, trestbps, chol, fbs, restecg,
                      thalach, exang, oldpeak, slope, ca, thal, num)
VALUES (999, 50, 1, 9, 120, 200, 0, 0, 150, 0, 1.0, 1, 0, 3, 0)
"""
try:
    conn.execute(bad_row)
except Exception as e:
    print(type(e).__name__, "-", e)
conn.rollback()

IntegrityError - FOREIGN KEY constraint failed


### First SQL look at the data

Disease rate by chest-pain type:

In [11]:
pd.read_sql("""
SELECT chest_pain_type,
       COUNT(*)                   AS patients,
       ROUND(AVG(target) * 100, 1) AS disease_pct
FROM patients_labeled
GROUP BY chest_pain_type
ORDER BY disease_pct DESC
""", conn)

,chest_pain_type,patients,disease_pct
0,asymptomatic,142,72.5
1,typical_angina,23,30.4
2,non_anginal_pain,83,21.7
3,atypical_angina,49,18.4


In [12]:
conn.close()

## 6. Summary

- **Missing values:** 6 rows with missing `ca`/`thal` dropped → **297 patients** (160 no disease, 137 disease; balance essentially unchanged).
- **Types:** all integer except `oldpeak`.
- **Encoding:** `cp`, `restecg`, `slope`, `thal` one-hot encoded → `heart_encoded.csv` (297 × 24), ready for modelling.
- **Storage:** `heart_clean.csv` and `heart.db` in `data/processed/`; the database enforces the data dictionary through keys and CHECK constraints.
- **Early signal:** patients with *asymptomatic* chest pain have by far the highest disease rate, a counter-intuitive pattern worth exploring in the EDA.

**Next (Week 3):** exploratory data analysis and SQL queries.